In [ ]:
import pandas as pd
import numpy as np

print("1. Initializing ACLED Conflict Data Cleaning Pipeline...")


# 1. LOADED RAW ACLED DATA
# ==========================================
# Assuming the raw ACLED file for Nigeria is downloaded and placed in data/raw/
try:
    df_acled = pd.read_csv("../data/raw/acled_nigeria.csv", low_memory=False)
    print(f"Successfully loaded {len(df_acled)} raw conflict events.")
except FileNotFoundError:
    print("Warning: Raw ACLED data not found. Please place 'acled_nigeria.csv' in data/raw/")
    # Creating a temporary empty DataFrame to allow the script to execute without failing
    df_acled = pd.DataFrame(columns=['year', 'admin1', 'event_type', 'fatalities'])

# ==========================================
# 2. FILTER TEMPORAL SCOPE 
# ==========================================
# Filter for events occurring specifically in 2022 to align temporally 
# with the BudgIT and Afrobarometer Round 9 datasets.
if not df_acled.empty:
    df_acled = df_acled[df_acled['year'] == 2022]

# ==========================================
# 3. STRING STANDARDIZATION & CASE FIXES
# ==========================================
# ACLED stores state names in the 'admin1' column. 
# Forcing uppercase and stripping whitespace prevents mismatch errors during dataset merges.
if not df_acled.empty:
    df_acled['admin1'] = df_acled['admin1'].astype(str).str.strip().str.upper()
    
    # Address standard naming discrepancies (e.g., standardizing the Federal Capital Territory)
    state_mapping = {
        'FEDERAL CAPITAL TERRITORY': 'FCT ABUJA',
        'ABUJA': 'FCT ABUJA'
    }
    df_acled['admin1'] = df_acled['admin1'].replace(state_mapping)
    
    # Rename for consistency across the master pipeline
    df_acled.rename(columns={'admin1': 'State'}, inplace=True)

# ==========================================
# 4. AGGREGATE METRICS BY SUBNATIONAL STATE
# ==========================================
print("2. Aggregating Conflict Events and Fatalities by State...")
if not df_acled.empty:
    acled_summary = df_acled.groupby('State').agg(
        conflict_events=('State', 'count'),       # Total number of recorded incidents
        fatalities=('fatalities', 'sum')          # Total death toll from incidents
    ).reset_index()
else:
    acled_summary = pd.DataFrame(columns=['State', 'conflict_events', 'fatalities'])

# ==========================================
# 5. EXPORT PROCESSED INDICATOR
# ==========================================
acled_summary.to_csv("../data/processed/acled_state_summary.csv", index=False)
print("SUCCESS: ACLED data cleaned, aggregated, and exported to 'data/processed/acled_state_summary.csv'")

if not acled_summary.empty:
    print("\nTop 5 States by Conflict Events (2022):")
    print(acled_summary.sort_values(by='conflict_events', ascending=False).head())

1. Initializing ACLED Conflict Data Cleaning Pipeline...
2. Aggregating Conflict Events and Fatalities by State...
SUCCESS: ACLED data cleaned, aggregated, and exported to 'data/processed/acled_state_summary.csv'
